# Session 2: the mathematics of a strong acid - strong base titration
**A. Real-world problem.** You must pick an indicator (a dye that changes colour) for a
titration, and you must turn a burette reading into a concentration. Both need numbers:
the pH at every point, the volume at the equivalence point, and the pH range of the dye.

**Safety.** Real acids, bases and indicators need supervision; this is a simulation.

**Plan.** (1) pH, pOH and pH + pOH = 14, (2) the mole balance Ca Va = Cb Vb, (3) pH before,
at and after equivalence, (4) choosing an indicator, (5) dilution C1 V1 = C2 V2.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
KW = 1e-14        # ion product of water at 25 degrees C, (mol/L)^2
# approximate, well-known pH ranges over which the dye changes colour
INDICATORS = {"methyl orange": (3.1, 4.4), "phenolphthalein": (8.2, 10.0)}

## Step 1: pH, pOH and the water constant
`pH = -log10[H+]` and `pOH = -log10[OH-]`. Water always keeps `[H+][OH-] = Kw = 1e-14`
at 25 C. Taking -log10 of both sides gives `pH + pOH = 14`.

In [ ]:
def ph_of_h(h):
    return -np.log10(h)


def poh_of_oh(oh):
    return -np.log10(oh)


def poh_from_ph(ph):
    """pOH from pH at 25 C: pH + pOH = 14."""
    return 14.0 - ph


def h_of_ph(ph):
    return 10.0 ** (-ph)

## Step 2: moles balance, and the three regions
Volumes are in mL and concentrations in mol/L, so `C * V` is in millimoles (mmol).
Net leftover concentration `x = (Ca Va - Cb Vb) / (Va + Vb)`: positive means leftover acid,
negative leftover base. Including water, `x = [H+] - Kw/[H+]`, and solving the quadratic gives
`[H+] = (x + sqrt(x^2 + 4 Kw)) / 2`. At equivalence `x = 0` so `[H+] = 1e-7` and pH = 7.

In [ ]:
def net_excess(Ca, Va, Cb, Vb):
    """Leftover strong acid (+) or strong base (-) in mol/L after adding Vb mL of base."""
    return (Ca * Va - Cb * Vb) / (Va + Vb)


def h_from_excess(x):
    """[H+] (mol/L) from the net excess x, numerically stable for both signs."""
    if x >= 0:
        return (x + np.sqrt(x * x + 4 * KW)) / 2
    return 2 * KW / (-x + np.sqrt(x * x + 4 * KW))


def strong_ph(Ca, Va, Cb, Vb):
    return float(-np.log10(h_from_excess(net_excess(Ca, Va, Cb, Vb))))


def equivalence_volume(Ca, Va, Cb):
    """Ca * Va = Cb * Ve  ->  Ve = Ca * Va / Cb  (mL)."""
    return Ca * Va / Cb


def unknown_concentration(Cb, Ve, Va):
    """Concentration of the acid in the flask (mol/L) from the equivalence volume."""
    return Cb * Ve / Va


def volume_at_ph(ph, Ca, Va, Cb):
    """Inverse of the curve: the base volume (mL) at which the flask has this pH.
    From Ca Va - Cb Vb = x (Va + Vb) with x = [H+] - Kw/[H+]."""
    h = h_of_ph(ph)
    x = h - KW / h
    return (Ca * Va - x * Va) / (Cb + x)

## Step 3: choose an indicator
A dye changes colour over a pH range (lo, hi). The colour change happens at the volume where
the flask pH is inside that range. The titration error is how far that volume is from the
true equivalence volume, as a percentage. A good indicator has its whole range on the steep
part, so every volume at which the colour could change is within a small error.

In [ ]:
def indicator_error_percent(lo, hi, Ca, Va, Cb):
    """Largest percentage error in the endpoint volume over the dye's colour range."""
    ve = equivalence_volume(Ca, Va, Cb)
    return max(abs(volume_at_ph(p, Ca, Va, Cb) - ve) for p in (lo, hi)) / ve * 100.0


def suitable(name, Ca, Va, Cb, tol_percent=2.0):
    lo, hi = INDICATORS[name]
    return indicator_error_percent(lo, hi, Ca, Va, Cb) <= tol_percent


def lowest_concentration(name, Va=25.0, tol_percent=2.0):
    """Smallest equal acid and base concentration (mol/L) for which the dye keeps its error
    within tol_percent. Found by bisection on log10(C)."""
    lo_c, hi_c = -6.0, 0.0           # log10 of concentration
    if not suitable(name, 10 ** hi_c, Va, 10 ** hi_c, tol_percent):
        return float("nan")
    for _ in range(60):
        mid = (lo_c + hi_c) / 2
        if suitable(name, 10 ** mid, Va, 10 ** mid, tol_percent):
            hi_c = mid
        else:
            lo_c = mid
    return 10 ** hi_c

## Step 4: dilution
Adding solvent does not change the moles of dissolved acid: `C1 V1 = C2 V2`.

In [ ]:
def final_volume(C1, V1, C2):
    """Volume (same unit as V1) needed to dilute V1 of concentration C1 down to C2."""
    return C1 * V1 / C2


def titration_curve(Ca, Va, Cb, vmax, n=601):
    vb = np.linspace(0.0, vmax, n)
    return vb, np.array([strong_ph(Ca, Va, Cb, v) for v in vb])

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    Va, Cb = 25.0, 0.1
    fig, ax = plt.subplots(figsize=(7, 4.6))
    for c, col in ((0.1, "tab:blue"), (0.01, "tab:orange"), (0.001, "tab:green")):
        ve = equivalence_volume(c, Va, c)
        vb, ph = titration_curve(c, Va, c, 2 * ve)
        ax.plot(vb / ve, ph, color=col, lw=2, label=f"{c:g} mol/L acid and base")
        print(f"C = {c:g}: pH at 0 mL {ph[0]:.2f}, at Ve {strong_ph(c, Va, c, ve):.2f}, "
              f"at 1.1 Ve {strong_ph(c, Va, c, 1.1 * ve):.2f}")
    for name, (lo, hi), col in (("methyl orange", INDICATORS["methyl orange"], "salmon"),
                                ("phenolphthalein", INDICATORS["phenolphthalein"], "violet")):
        ax.axhspan(lo, hi, color=col, alpha=0.3)
        ax.text(0.03, (lo + hi) / 2, name, va="center", fontsize=8)
    ax.set_xlabel("base added / equivalence volume (no unit)")
    ax.set_ylabel("pH of the flask (no unit)")
    ax.set_title("Strong acid - strong base: coloured bands are indicator ranges")
    ax.set_ylim(0, 14)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(OUT / "strong_curves_indicators.png", dpi=120)

    cs = np.logspace(-3, 0, 60)
    fig2, ax2 = plt.subplots(figsize=(6.5, 4.2))
    for name in INDICATORS:
        lo, hi = INDICATORS[name]
        err = [indicator_error_percent(lo, hi, c, Va, c) for c in cs]
        ax2.loglog(cs, err, lw=2, label=name)
    ax2.axhline(2.0, color="red", ls="--", label="2% limit")
    ax2.set_xlabel("acid and base concentration (mol/L)")
    ax2.set_ylabel("endpoint error (%)")
    ax2.set_title("Dilute solutions need a better matched indicator")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "indicator_error.png", dpi=120)
    for name in INDICATORS:
        print(f"{name}: error at 0.1 mol/L = {indicator_error_percent(*INDICATORS[name], 0.1, Va, 0.1):.3f}%, "
              f"at 0.001 mol/L = {indicator_error_percent(*INDICATORS[name], 0.001, Va, 0.001):.2f}%, "
              f"lowest concentration within 2% = {lowest_concentration(name):.4g} mol/L")
    print("dilute 10 mL of 1.0 M to 0.1 M needs final volume", final_volume(1.0, 10.0, 0.1), "mL")

## Coding tasks
1. **Easy.** Use `strong_ph` to print the pH of 0.01 mol/L HCl before any base is added, and the pH after 26 mL of 0.01 mol/L NaOH is added to 25 mL of the acid. Predict both by hand first.
2. **Stretch.** Find, with `indicator_error_percent`, the endpoint error of phenolphthalein for 0.02 mol/L acid and base, and decide whether it meets a 2 percent limit.
3. **Explain.** Explain why the endpoint error for methyl orange is dominated by the low end of its range (pH 3.1) rather than the high end (pH 4.4). (Hint: compare the leftover acid at each pH.)

## Assessment
- **A. Conceptual.** Why does pH + pOH = 14 hold for every aqueous solution at 25 C, even for a solution of a base?
- **B. Numerical.** A 25.00 mL sample of acid needs 18.60 mL of 0.1000 mol/L NaOH. What is the acid concentration? What volume of water must be added to 10.00 mL of that acid to dilute it to a quarter of its concentration?
- **C. Graph and simulation interpretation.** In the plot of three curves, what do the three curves have in common and how do they differ? What does that mean for finding the equivalence point of a very dilute acid?
- **D. Design and optimisation challenge.** Design: which indicator (methyl orange or phenolphthalein) is suitable for a titration of 0.05 mol/L HCl with 0.05 mol/L NaOH, with an endpoint error of 2 percent allowed? For phenolphthalein, below roughly what concentration does it stop being suitable?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("pH + pOH = 14 for pH 3.7", abs(L.poh_from_ph(3.7) + 3.7 - 14.0) < 1e-12))
    checks.append(("pOH of 1e-3 mol/L OH- is 3, so pH is 11", abs(L.poh_of_oh(1e-3) - 3.0) < 1e-12 and abs(L.poh_from_ph(3.0) - 11.0) < 1e-12))
    checks.append(("0.1 M HCl titrated with 0.1 M NaOH: pH 1 at the start", abs(L.strong_ph(0.1, 25.0, 0.1, 0.0) - 1.0) < 1e-6))
    ve = L.equivalence_volume(0.1, 25.0, 0.1)
    checks.append(("pH is 7 at the equivalence volume", abs(L.strong_ph(0.1, 25.0, 0.1, ve) - 7.0) < 1e-6))
    checks.append(("pH is high (above 11) well after equivalence", L.strong_ph(0.1, 25.0, 0.1, 30.0) > 11.0))
    # worked example: 25.00 mL of unknown HCl needs 20.00 mL of 0.1000 M NaOH
    c = L.unknown_concentration(0.1, 20.0, 25.0)
    checks.append((f"worked example: 20.00 mL of 0.1000 M NaOH for 25.00 mL of acid gives {c:.4f} mol/L and pH {-np.log10(c):.3f}",
                   abs(c - 0.08) < 1e-12 and abs(-np.log10(c) - 1.097) < 1e-3))
    # hand calculation: 0.1 mL before equivalence, leftover 0.01 mmol in 49.9 mL
    checks.append(("hand calculation: 24.9 mL gives pH = -log10(0.01/49.9) = 3.698",
                   abs(L.strong_ph(0.1, 25.0, 0.1, 24.9) - (-np.log10(0.01 / 49.9))) < 1e-5))
    checks.append(("hand calculation: 25.1 mL gives pH = 14 + log10(0.01/50.1) = 10.300",
                   abs(L.strong_ph(0.1, 25.0, 0.1, 25.1) - (14 + np.log10(0.01 / 50.1))) < 1e-5))
    checks.append(("inverse function: volume_at_ph undoes strong_ph",
                   all(abs(L.volume_at_ph(L.strong_ph(0.1, 25.0, 0.1, v), 0.1, 25.0, 0.1) - v) < 1e-6 for v in (5.0, 20.0, 24.9, 25.1, 30.0))))
    checks.append(("both dyes work for 0.1 M (error under 2 percent)", L.suitable("methyl orange", 0.1, 25.0, 0.1) and L.suitable("phenolphthalein", 0.1, 25.0, 0.1)))
    checks.append(("for 0.001 M neither dye is good enough", not L.suitable("methyl orange", 0.001, 25.0, 0.001) and not L.suitable("phenolphthalein", 0.001, 25.0, 0.001)))
    checks.append(("phenolphthalein tolerates a more dilute solution than methyl orange here",
                   L.lowest_concentration("phenolphthalein") < L.lowest_concentration("methyl orange")))
    checks.append(("dilution C1 V1 = C2 V2: 10 mL of 1.0 M to 0.1 M needs 100 mL", abs(L.final_volume(1.0, 10.0, 0.1) - 100.0) < 1e-12))
    checks.append(("finite at the exact boundary (Vb = Ve) and for a huge excess", bool(np.isfinite(L.strong_ph(0.3, 10.0, 0.3, 10.0)) and np.isfinite(L.strong_ph(0.1, 25.0, 0.5, 500.0)))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
